In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from delta.tables import *

In [0]:
dbutils.widgets.text("src","")
src_value=dbutils.widgets.get("src")
print(src_value)

In [0]:
fact_config = {
    "orders": {
        "source_object": "silver_orders",
        "source_schema": "silver_ecommerce",
        "target_object": "FactOrders",
        "target_schema": "gold_ecommerce",
        "key_cols": ["order_id"],
        "cdc_col": "modified_date"
    },

    "payments": {
        "source_object": "silver_payments",
        "source_schema": "silver_ecommerce",
        "target_object": "FactPayments",
        "target_schema": "gold_ecommerce",
        "key_cols": ["payment_id"],
        "cdc_col": "modified_date"
    }
}

### **Parameters**

In [0]:
config=fact_config[src_value]
print(config)

In [0]:
key_col_list=config["key_cols"]

cdc_col=config["cdc_col"]

back_dated_col=''

source_schema=config["source_schema"]

source_object=config["source_object"]

target_schema=config["target_schema"]

target_object=config["target_object"]




### **Incremental Loading**

In [0]:
if len(back_dated_col)==0:
    if spark.catalog.tableExists(f"{target_schema}.{target_object}"):
        last_load=spark.sql(f"select max({cdc_col}) from {target_schema}.{target_object}").collect()[0][0]
    else:
        last_load='1900-01-01 00:00:00'
else:
    last_load=back_dated_col
print(last_load)

In [0]:
df_source=spark.sql(f"select * from {source_schema}.{source_object} where {cdc_col}>'{last_load}'")


In [0]:
display(df_source)

In [0]:
if src_value=='orders':
    dim_customers=spark.read.table("gold_ecommerce.dimcustomers")
    dim_products=spark.read.table("gold_ecommerce.dimproducts")
    df_fact_orders=df_source.join(dim_customers.select("DIMCustomersKey","customer_id"),df_source.customer_id==dim_customers.customer_id,'left').join(dim_products.select("DIMProductsKey","product_id"),df_source.product_id==dim_products.product_id,'left')
    fact_orders=df_fact_orders.select("order_id","DIMCustomersKey","DIMProductsKey","order_amount","order_status","order_timestamp","quantity","modified_date","processed_date")
    fact_orders.printSchema()
elif src_value=='payments':
    fact_orders=spark.read.table("gold_ecommerce.factorders")
    fact_payments=df_source.join(fact_orders.select(col("order_id").alias("fo_order_id")),df_source.order_id==col("fo_order_id"),'left')
    fact_payments=fact_payments.select("payment_id","order_id","payment_method","payment_status","payment_amount","payment_timestamp","modified_date","processed_date")

In [0]:
if src_value=='orders':
    if spark.catalog.tableExists(f"{target_schema}.{target_object}"):
        df_target=DeltaTable.forName(spark,f"{target_schema}.{target_object}")
        df_target.alias("trg").merge(fact_orders.alias("src"),f"trg.order_id=src.order_id")\
        .whenMatchedUpdateAll(condition=f"src.{cdc_col}>trg.{cdc_col}").whenNotMatchedInsertAll().execute()
    else:
        fact_orders.write.mode("overwrite").format("delta").saveAsTable(f"{target_schema}.{target_object}")
elif src_value=='payments':
    if spark.catalog.tableExists(f"{target_schema}.{target_object}"):
        df_target=DeltaTable.forName(spark,f"{target_schema}.{target_object}")
        df_target.alias("trg").merge(fact_payments.alias("src"),f"trg.payment_id=src.payment_id")\
        .whenMatchedUpdateAll(condition=f"src.{cdc_col}>trg.{cdc_col}").whenNotMatchedInsertAll().execute()
    else:
        fact_payments.write.mode("overwrite").format("delta").saveAsTable(f"{target_schema}.{target_object}")
    